In [ ]:
import os
# os.environ["TRANSFORMERS_NO_ACCELERATE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1" #离线加载

In [ ]:
import jieba
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from torch.distributed.pipelining import pipeline
from umap import UMAP
import numpy as np
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
import torch
print(torch.cuda.is_available())
# torch.set_default_device('cuda')
torch.__version__

In [ ]:
# 导入数据
# with open('陕西-3.txt', 'r', encoding='utf-8') as f:
#     docs=f.readlines()
with open('师德师风切词.txt', 'r', encoding='utf-8') as f:
    docs=f.readlines()
print("文本数：",len(docs))
print("第一条：",docs[0])
vectorizer_model = None

In [54]:
# set HF_ENDPOINT=https://hf-mirror.com
# 0 sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
# embedding_model=SentenceTransformer("C:\\Users\\xule\\.cache\\huggingface\\hub\\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2\\snapshots\\86741b4e3f5cb7765a600d3a3d55a0f6a6cb443d")
# embedding_model=SentenceTransformer("C:\\Users\\xule\\.cache\\huggingface\\hub\\models--KaLM-Embedding--KaLM-embedding-multilingual-mini-instruct-v2.5\\snapshots\\753c6fe26abc20a32aeb162003aa03457d15db2f")
from transformers import pipeline
from sentence_transformers import SentenceTransformer
# print(transformers.__version__)

from transformers import AutoModel, AutoTokenizer

# model_name = "bert-base-chinese"
# model = AutoModel.from_pretrained(model_name).to("cuda")
# tokenizer = AutoTokenizer.from_pretrained(model_name)
#
# embedding_model = pipeline("feature-extraction", model=model, tokenizer=tokenizer,device=0)

from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('bert-base-chinese', device='cuda')
embeddings=np.load('emb_师德师风.npy')
# print(1)

In [55]:
umap_model=UMAP(
    n_neighbors=5,  #参考周围的5个连续点
    n_components=5, #将高维数据降维到5维
    min_dist=0.0,   #嵌入点之间的最小的有效距离，值越小会导致越密集的聚类（会减少离群值），默认为0.1
    metric='cosine',    #余弦距离，默认会使用欧几里得距离
    random_state=86 #设置随机数种子
)

In [56]:
hdbscan_model=HDBSCAN(
    min_cluster_size=15,
    min_samples=5,  #减少该值可以减少离群值，默认等于min_cluster_size
    metric='euclidean',
    # prediction_data=True,   #若计算文本属于每一个主题的概率，则必须为True
)   #每个类中至少包含5条

In [57]:
# def tokenize_zh(text):  #切词
#     words=jieba.cut(text)
#     return words
#
# # 若导入的是切词后的文件，则下一条语句无需填写参数
# vectorizer_model=CountVectorizer(tokenizer=tokenize_zh)
vectorizer_model=CountVectorizer(stop_words=['二零一二年','二零一三年','二零一四年','二零一五年','二零一六年','二零一七年','二零一八年','二零一九年','二零二零年','二零二一年','二零二二年','二零二三年','二零二四年','二零二五年','二零二六年'])

if torch.cuda.is_available():
    print(f"当前可用 GPU: {torch.cuda.device_count()}")
    print(f"GPU 0 名称: {torch.cuda.get_device_name(0)}")

当前可用 GPU: 1
GPU 0 名称: NVIDIA GeForce RTX 4060 Laptop GPU


In [58]:
print("embedding_model 设备:", embedding_model.device)

embedding_model 设备: cuda:0


In [59]:
topic_model=BERTopic(
    embedding_model=embedding_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    # min_topic_size=5, #每个主题至少包含5条，有min_cluster_size就不用设置
    # nr_topics='auto',   #自动合并主题
    # calculate_probabilities=True,   #计算属于每一个主题的概率
)
# print(embeddings.shape)
# print(len(docs))
topics, probs=topic_model.fit_transform(docs)
topic_info=topic_model.get_topic_info()
topic_info
topic_info.to_csv('topic_info.csv', index=False, encoding='utf-8-sig')

In [60]:
print(len(topics),topics[:10])
print(len(probs),probs[:10])

848 [-1, -1, 1, 0, 0, 2, 1, 8, 4, 10]
848 [0.         0.         1.         1.         1.         0.69798821
 0.98252333 0.9187863  1.         1.        ]


In [61]:
# topic_model.set_topic_labels({
#     0:'',
#     1:'',
#     2:''
# })
# topic_info=topic_model.get_topic_info()
# topic_info
# topic_model.visualize_barchart(custom_labels=True)
topic_model.visualize_barchart()

In [62]:
topic_model.visualize_topics()

In [63]:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings, hide_document_hover=True)

In [64]:
import plotly.express as px
hierarchical_topics = topic_model.hierarchical_topics(docs)
# topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

# 为每个轨迹随机分配颜色（或使用预定义颜色列表）
colors = ['red', 'blue', 'green', 'purple', 'orange', 'cyan']
for i, trace in enumerate(fig.data):
    trace.line.color = colors[i % len(colors)]

fig.show()
fig.write_html('topic_hierarchy.html')

100%|██████████| 15/15 [00:00<00:00, 306.11it/s]


In [65]:
with open('师德师风时间.txt','r',encoding='utf-8') as file:
    lines=file.readlines()
    timestamps=[int(line.strip()) for line in lines]
print(len(timestamps),timestamps[:10])

848 [2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021]


In [66]:
topics_over_time=topic_model.topics_over_time(docs,timestamps,global_tuning=False, evolution_tuning=False) # global_tuning=False, evolution_tuning=False
topic_model.visualize_topics_over_time(topics_over_time)

In [67]:
topic_docs = topic_model.get_document_info(docs)
topic_docs.to_csv('师德师风聚类结果.csv')